# T3a — Creación de la franja de edad

**Autor:** Jorge Martín 

**Proyecto:** P1 — Análisis exploratorio del Titanic

**Asignatura:** Python para IA — 2º DAM

## Objetivo

Crear una nueva columna llamada `franja_edad` a partir de la columna `age` del dataset Titanic.

Las franjas de edad establecidas en el proyecto son:

- `0-12`
- `13-17`
- `18-59`
- `60+`

La nueva columna se creará utilizando NumPy y operaciones vectorizadas, sin utilizar bucles `for` sobre las filas.

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset desde la carpeta Data del repositorio.
# Si no se encuentra (por ejemplo al abrir el notebook en Colab), se descarga
# de la misma fuente original, así que los datos son idénticos.
try:
    df = pd.read_csv("../Data/titanic.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

df.shape

(891, 15)

Aplicamos la base común del contrato para trabajar con las edades ya sin nulos (la mediana imputada, ver T2a).

In [2]:
# --- BASE COMÚN (tratamiento de nulos acordado en el contrato, ver T2a) ---
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns="deck")

In [3]:
# Mostramos las primeras 10 edades
df["age"].head(10)

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5    28.0
6    54.0
7     2.0
8    27.0
9    14.0
Name: age, dtype: float64

In [4]:
# Definimos las condiciones para cada franja de edad.
# Usamos límites contiguos (< 13, < 18, < 60) para que ninguna edad con decimales
# (por ejemplo 12,5 o 59,5) se quede fuera de todas las franjas.
edad = df["age"]
condiciones = [
    edad < 13,
    (edad >= 13) & (edad < 18),
    (edad >= 18) & (edad < 60),
    edad >= 60,
]

# Definimos el nombre de cada franja
valores = ["0-12", "13-17", "18-59", "60+"]

# Creamos la nueva columna con NumPy, de forma vectorizada (sin bucles).
# El valor por defecto es texto para que toda la columna sea del mismo tipo.
df["franja_edad"] = np.select(condiciones, valores, default="Desconocido")

In [5]:
# Mostramos la edad junto con la nueva franja
df[["age", "franja_edad"]].head(20)

,age,franja_edad
0,22.0,18-59
1,38.0,18-59
2,26.0,18-59
3,35.0,18-59
4,35.0,18-59
5,28.0,18-59
6,54.0,18-59
7,2.0,0-12
8,27.0,18-59
9,14.0,13-17


In [6]:
# Contamos cuántos pasajeros hay en cada franja
df["franja_edad"].value_counts()

franja_edad
18-59    752
0-12      69
13-17     44
60+       26
Name: count, dtype: int64

In [7]:
# Comprobamos las categorías existentes
df["franja_edad"].unique()

<StringArray>
['18-59', '0-12', '13-17', '60+']
Length: 4, dtype: str

## Hallazgo

La columna `franja_edad` agrupa a los 891 pasajeros en cuatro franjas sin dejar ninguno fuera: 69 tienen entre 0 y 12 años, 44 entre 13 y 17, 752 entre 18 y 59, y 26 tienen 60 o más. La franja 18-59 concentra el 84% de los pasajeros, en parte porque incluye a los 177 pasajeros cuya edad se rellenó con la mediana (28 años).